<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/forest_area_class_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  Forest Threshold & Type Analysis: Case Study
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [ ]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────


In [ ]:
# ── CELL 2: conect to drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
print('✅ connected to drive')

In [1]:
# ── CELL 3: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess


if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main
print('✅ Ready')

/content/Biochar_forest_estimation
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 2.15 KiB | 440.00 KiB/s, done.
From https://github.com/MElsdk-lab/Biochar_forest_estimation
   396c0c9..bf39470  main       -> origin/main
HEAD is now at bf39470 Add forest class definitions to data_config.py
✅ Ready


In [3]:
# ── CELL 4: Authenticate and initialize Earth Engine & Load Datasets  ─────────────────────
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too

Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')

treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

#Load GLC FCS30D Year 2000

glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()


✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded
✅ treecover2000 masked


In [6]:
# ── CELL 5: import libraries and data from data_config ─────────────────────

import ee
import geemap
import pandas as pd
import time
from data_config import (FAO_LSIB_REGION, us_state_names, get_all_countries, build_country_lookup,
                         forest_bins, country_thresholds, state_thresholds, forestClasses
                        )


from gee_functions import (
    # Section 1: Forest area by threshold — countries
    prepare_forest_collection,
    export_forest_area,
    # Section 2: Forest area by threshold — states
    prepare_states_forest_collection,
    export_states_forest_area,
    # Section 3: Forest cover bins — countries
    get_forest_cover_bins_one_country,
    export_forest_cover_bins_all_countries,
    # Section 4: Forest cover bins — states
    get_forest_cover_bins_one_state,
    export_forest_cover_bins_all_states,
    # Section 5: GLC forest type area — countries
    get_forest_cover_area_type_country,
    export_forest_cover_area_type_all_countries,
    # Section 6: GLC forest type area — states
    get_forest_cover_area_type_state,
    export_forest_cover_area_type_all_states,
  )

import gee_functions
gee_functions.treecover2000_masked = treecover2000_masked
gee_functions.datamask             = datamask
gee_functions.glc_2000             = glc_2000


print('✅ GEE objects injected into gee_functions')
print('✅ All GEE functions imported')
print('✅ Libraries imported')
print('✅ Data config loaded')

✅ GEE objects injected into gee_functions
✅ All GEE functions imported
✅ Libraries imported
✅ Data config loaded


In [7]:
# ── CELL 6: Run Exports — United States All States ────────────────────────────

print('── Submitting forest area by threshold task ──')
state_threshold_task = export_states_forest_area(us_state_names, state_thresholds)

print('\n── Submitting forest cover bins task ──')
state_bins_task = export_forest_cover_bins_all_states(us_state_names, forest_bins)

print('\n── Submitting GLC forest type area task ──')
state_glc_task = export_forest_cover_area_type_all_states(us_state_names, forestClasses)

── Submitting forest area by threshold task ──
✅ Single export task submitted: states_forest_area_all_thresholds

── Submitting forest cover bins task ──
✅ Single export task submitted: forest_cover_bins_all_states

── Submitting GLC forest type area task ──
✅ Single export task submitted: forest_cover_area_type_all_states


In [ ]:
# ── CELL 12: Push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in globals():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)